# Narrow AI vs General AI: a hands-on look

Companion notebook for the article
[Narrow AI vs General AI](../../../docs/01-AI-FUNDAMENTALS/01-introduction-to-artificial-intelligence/04-narrow-ai-vs-general-ai.md).

**What you will learn**

1. What a *narrow* AI looks like in code: one model, trained for one job.
2. How well it does that job.
3. What happens when you give it something outside its job.
4. A simple engineering trick that lets a narrow system *notice* it is out of its depth.
5. Why that trick is still very far from *general* intelligence.

Everything runs locally in a few seconds. No API keys, no downloads. All the code is in this notebook.

## Step 1: Imports and settings

We need numpy (arrays of numbers) and a few pieces of scikit-learn. We also fix a few settings:

- `SEED = 42` makes every random step the same on every run, so you get the same numbers as the article.
- Each digit image is an 8x8 grid of pixels.
- Pixel values go from 0 (blank) to 16 (full ink).

In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

SEED = 42
IMAGE_SHAPE = (8, 8)   # each digit is an 8x8 grid of pixels
MAX_INK = 16.0         # pixel values go from 0 (blank) to 16 (full ink)

print("Ready. Random seed:", SEED)

Ready. Random seed: 42


## Step 2: Look at the data

We use scikit-learn's built-in **digits** dataset: 1,797 small 8x8 pictures of handwritten digits 0 to 9.
Each picture is 64 numbers (pixel brightness from 0 to 16). `load_data` keeps 25% aside as a test set the model never sees during training.

In [2]:
def load_data(seed: int = SEED):
    """Return X_train, X_test, y_train, y_test for the 8x8 digits dataset."""
    digits = load_digits()
    return train_test_split(
        digits.data, digits.target, test_size=0.25, random_state=seed, stratify=digits.target
    )


X_train, X_test, y_train, y_test = load_data()
print("Training images:", len(X_train))
print("Test images:    ", len(X_test))
print("Labels the model can ever output:", sorted(int(label) for label in set(y_train)))

Training images: 1347
Test images:     450
Labels the model can ever output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


To *see* an image without any plotting library, `ascii_image` draws it with text characters: `#` is full ink, `:` and `.` are lighter, a space is blank.

In [3]:
def ascii_image(image: np.ndarray) -> str:
    """Draw an 8x8 image with text characters so it shows up in any output."""
    shades = " .:#"
    grid = np.asarray(image, dtype=float).reshape(IMAGE_SHAPE)
    rows = []
    for row in grid:
        rows.append("".join(shades[min(int(v / MAX_INK * (len(shades) - 1) + 0.5), 3)] * 2 for v in row))
    return "\n".join(rows)


print(f"Test image #0 (true label: {y_test[0]})")
print(ascii_image(X_test[0]))

Test image #0 (true label: 1)
    ::####::    
    ######      
    ####::      
    ####::      
  ..####::      
    ####::      
    ####::      
    ..####      


Notice the list of labels: **0 to 9 and nothing else**. Whatever we show this model later, it can only answer with one of those ten digits.
That fixed set of possible answers is the first sign of a narrow AI.

## Step 3: Train the narrow model

We train a logistic regression classifier. It learns which pixel patterns go with which digit.

In [4]:
def train_digit_model(X_train: np.ndarray, y_train: np.ndarray) -> LogisticRegression:
    """Train a narrow AI: a classifier that only knows the labels 0-9."""
    model = LogisticRegression(max_iter=2000)
    model.fit(X_train, y_train)
    return model


model = train_digit_model(X_train, y_train)
accuracy = model.score(X_test, y_test)
print(f"Accuracy on unseen digits: {accuracy:.1%}")

Accuracy on unseen digits: 96.2%


Around 96% on digits it has never seen. Inside its one job, this model is genuinely good.
That is what narrow AI is: **strong performance on a specific, well-defined task**.

**Quick check.** An `assert` line does nothing when the statement is true and stops with an error when it is false. These two lines pin down the two facts above: the model is good at its job, and its only possible answers are the digits 0 to 9.

In [5]:
assert accuracy > 0.9
assert list(model.classes_) == list(range(10))
print("Checks passed: accurate on its one job, and only knows the labels 0-9.")

Checks passed: accurate on its one job, and only knows the labels 0-9.


## Step 4: Ask it about a few real digits

`predict_with_confidence` returns the predicted digit and the model's probability for that answer (0 to 1).
`predict_proba` gives one probability per known label, and the ten probabilities always add up to 1. There is **no slot for "not a digit"**.

In [6]:
def predict_with_confidence(model: LogisticRegression, image: np.ndarray) -> tuple[int, float]:
    """Return (predicted digit, confidence between 0 and 1) for one 8x8 image."""
    row = np.asarray(image, dtype=float).reshape(1, -1)   # one row of 64 numbers
    probabilities = model.predict_proba(row)[0]            # 10 probabilities that sum to 1
    best = int(np.argmax(probabilities))
    return int(model.classes_[best]), float(probabilities[best])


for i in range(5):
    digit, confidence = predict_with_confidence(model, X_test[i])
    print(f"true={y_test[i]}  predicted={digit}  confidence={confidence:.2f}")

true=1  predicted=8  confidence=0.84
true=0  predicted=0  confidence=1.00
true=9  predicted=9  confidence=0.99
true=1  predicted=1  confidence=1.00
true=5  predicted=5  confidence=1.00


Four out of five are right with high confidence. Look at the first one, though: a **1** was read as an **8** with 0.84 confidence.
Even inside its own job, a narrow model makes mistakes, and its confidence does not always warn you.

## Step 5: Step outside the job

Now we build four 8x8 "images" that are **not digits**: random noise, a blank page, the letter X, and a checkerboard.
They are valid input for the model (64 numbers in the right range), but a person would say "that's not a digit".

In [7]:
def make_out_of_domain_inputs(seed: int = SEED) -> dict[str, np.ndarray]:
    """Build 8x8 'images' that are NOT handwritten digits."""
    rng = np.random.default_rng(seed)
    letter_x = np.zeros(IMAGE_SHAPE)
    for i in range(8):
        letter_x[i, i] = MAX_INK
        letter_x[i, 7 - i] = MAX_INK
    checkerboard = (np.indices(IMAGE_SHAPE).sum(axis=0) % 2) * MAX_INK
    return {
        "random noise": rng.uniform(0, MAX_INK, size=IMAGE_SHAPE),
        "blank page": np.zeros(IMAGE_SHAPE),
        "letter X": letter_x,
        "checkerboard": checkerboard,
    }

Let's see what the model says about each one.

In [8]:
odd_inputs = make_out_of_domain_inputs()
for name, image in odd_inputs.items():
    digit, confidence = predict_with_confidence(model, image)
    print(f"--- {name}: predicted digit {digit} with confidence {confidence:.2f}")
    print(ascii_image(image))

--- random noise: predicted digit 1 with confidence 0.69
::..##::  ##::::
  ....##::::....
::  ::::::..####
::....    ::::##
........  ....::
..::::..::::....
::  ..  ::::::::
..::    ::..::::
--- blank page: predicted digit 4 with confidence 0.11
                
                
                
                
                
                
                
                
--- letter X: predicted digit 8 with confidence 1.00
##            ##
  ##        ##  
    ##    ##    
      ####      
      ####      
    ##    ##    
  ##        ##  
##            ##
--- checkerboard: predicted digit 7 with confidence 0.76
  ##  ##  ##  ##
##  ##  ##  ##  
  ##  ##  ##  ##
##  ##  ##  ##  
  ##  ##  ##  ##
##  ##  ##  ##  
  ##  ##  ##  ##
##  ##  ##  ##  


Look closely at the results:

- The **letter X** is called an **8** with almost 100% confidence. The model is confidently wrong.
- **Noise** and the **checkerboard** also get a digit, with fairly high confidence.
- Only the **blank page** gets low confidence, and even then the model still has to pick a digit.

The model has no concept of "this is not a digit", because nobody trained it for that.
It does not *know* what it does not know. High confidence is **not** the same as being right.

**Quick check:** every odd input still gets a digit, and the letter X is misread with more than 90% confidence.

In [9]:
for image in odd_inputs.values():
    digit, _ = predict_with_confidence(model, image)
    assert digit in range(10)   # it always answers with a digit; "not a digit" is impossible

_, x_confidence = predict_with_confidence(model, odd_inputs["letter X"])
assert x_confidence > 0.9
print("Checks passed: always answers with a digit, and confidently wrong about the letter X.")

Checks passed: always answers with a digit, and confidently wrong about the letter X.


## Step 6: Try a completely different task

A general intelligence could switch tasks: read a sentence, do some arithmetic, describe a picture.
Let's hand our model a sentence and see what happens.

In [10]:
print("This model expects exactly", model.n_features_in_, "numbers (pixels) per input.")

sentence = "Is this restaurant review positive or negative?"
try:
    model.predict([[sentence]])
except ValueError as error:
    print("Given a sentence instead, it fails:")
    print("ValueError:", str(error).splitlines()[0])

This model expects exactly 64 numbers (pixels) per input.
Given a sentence instead, it fails:
ValueError: dtype='numeric' is not compatible with arrays of bytes/strings.Convert your data to numeric values explicitly instead.


It fails before it even starts: the model only accepts exactly 64 numbers.
Nothing it learned about digits transfers to reading text. To handle reviews you would need **a different model trained on different data**.
That is the everyday reality of narrow AI: one model per job.

## Step 7: Help the narrow system notice when it is out of its depth

We cannot make this model general, but we *can* add a guard that asks: **"Does this input look like anything I was trained on?"**

`DomainGuard` measures the distance from a new input to its closest training image.
If that distance is bigger than 99% of the distances between real training images, it flags the input as out-of-domain.

- `fit` uses the **2nd** nearest neighbour, because the 1st neighbour of a training image is the image itself (distance 0).
- `distance` finds how far a new image is from its closest training image.
- `is_in_domain` compares that distance with the threshold.

In [11]:
class DomainGuard:
    """A simple 'is this like my training data?' check."""

    def __init__(self, percentile: float = 99.0) -> None:
        self.percentile = percentile
        self.threshold = None
        self._nn = None

    def fit(self, X_train: np.ndarray) -> "DomainGuard":
        self._nn = NearestNeighbors(n_neighbors=2).fit(X_train)
        # distance from each training image to its closest *other* training image
        distances, _ = self._nn.kneighbors(X_train)
        self.threshold = float(np.percentile(distances[:, 1], self.percentile))
        return self

    def distance(self, image: np.ndarray) -> float:
        row = np.asarray(image, dtype=float).reshape(1, -1)
        distances, _ = self._nn.kneighbors(row, n_neighbors=1)
        return float(distances[0, 0])

    def is_in_domain(self, image: np.ndarray) -> bool:
        return self.distance(image) <= self.threshold

Now fit the guard on the training images and try it on real test digits and on the four odd inputs.

In [12]:
guard = DomainGuard(percentile=99).fit(X_train)
print(f"Distance threshold: {guard.threshold:.1f}")

in_domain_rate = np.mean([guard.is_in_domain(x) for x in X_test])
print(f"Real test digits accepted by the guard: {in_domain_rate:.1%}")
print()
for name, image in odd_inputs.items():
    verdict = "accept" if guard.is_in_domain(image) else "REJECT -> hand to a human"
    print(f"{name:13s} distance={guard.distance(image):5.1f}  {verdict}")

Distance threshold: 27.6


Real test digits accepted by the guard: 99.8%

random noise  distance= 58.4  REJECT -> hand to a human
blank page    distance= 50.3  REJECT -> hand to a human
letter X      distance= 59.9  REJECT -> hand to a human
checkerboard  distance= 78.4  REJECT -> hand to a human


The guard accepts almost all real digits and rejects all four odd inputs.

**Quick check:** the guard should let real digits through and stop every odd input.

In [13]:
assert in_domain_rate > 0.95
for name, image in odd_inputs.items():
    assert not guard.is_in_domain(image), name
print("Checks passed: real digits accepted, every odd input rejected.")

Checks passed: real digits accepted, every odd input rejected.


This is a common engineering pattern, and AI agents use the same idea when they **escalate to a human** or **decline a task outside their scope**.
But notice what it does *not* do: it does not teach the model anything about letters, noise or restaurant reviews.
The system is still narrow; it is just **honest about being narrow**.

## Step 8: What would "general" require?

To be *general*, a single system would need abilities this notebook cannot even attempt:

| Ability | Our digit model | Today's LLM-based agents (broad) | AGI (hypothetical) |
|---|---|---|---|
| Handle many unrelated tasks | No, one task | Yes, many text, code and image tasks | Yes, essentially all cognitive tasks |
| Learn a brand-new skill from a few examples, reliably | No, needs retraining | Partly (in-context learning), not reliably | Yes |
| Know when it is out of its depth | Only with an added guard | Partly; can still be confidently wrong | Yes |
| Pursue long, open-ended goals on its own | No | Limited; needs tools, guardrails and human oversight | Yes |

The middle column is a judgement call, not a measurement. Experts disagree about where today's systems sit, which is why the article describes them as *broad but not general*.

## Recap

- A **narrow AI** is trained for one job. Our digit model scores about 96% on unseen digits.
- Outside that job it **still answers**, sometimes with high confidence (the letter X became an 8).
- It cannot switch tasks at all: a sentence is not even a valid input.
- A **domain guard** lets a narrow system notice unfamiliar inputs and hand off, the same idea agents use when they escalate to a human.
- **General AI (AGI)** would handle essentially any cognitive task and learn new ones flexibly. It does not exist today, and when (or whether) it will is openly debated.

**Try next:** change `percentile` in `DomainGuard` to 90 or 99.9 and watch how many real digits get rejected.